I started by importing the file uploader from Colab. Then I uploaded the ACS social characteristics CSV, followed by its matching GeoJSON. After that, I did the same for the ADA barriers CSV and its GeoJSON. Just grabbed each file from my computer one by one.


In [ ]:
from google.colab import files

print("Upload ACS_5-Year_Social_Characteristics_DC_Ward.csv")
uploaded4 = files.upload()

print("Upload ACS_5-Year_Social_Characteristics_DC_Ward.geojson")
uploaded4 = files.upload()

print("Upload ADA_Barriers_in_the_Public_Right_of_Way.csv")
uploaded5 = files.upload()

print("Upload ADA_Barriers_in_the_Public_Right_of_Way.geojson")
uploaded5 = files.upload()

Upload ACS_5-Year_Social_Characteristics_DC_Ward.csv


Saving ACS_5-Year_Social_Characteristics_DC_Ward.csv to ACS_5-Year_Social_Characteristics_DC_Ward.csv
Upload ACS_5-Year_Social_Characteristics_DC_Ward.geojson


Saving ACS_5-Year_Social_Characteristics_DC_Ward.geojson to ACS_5-Year_Social_Characteristics_DC_Ward.geojson
Upload ADA_Barriers_in_the_Public_Right_of_Way.csv


Saving ADA_Barriers_in_the_Public_Right_of_Way.csv to ADA_Barriers_in_the_Public_Right_of_Way.csv
Upload ADA_Barriers_in_the_Public_Right_of_Way.geojson


Saving ADA_Barriers_in_the_Public_Right_of_Way.geojson to ADA_Barriers_in_the_Public_Right_of_Way.geojson


Then, I installed all the necessry libraries

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

I loaded the ADA barriers CSV into a dataframe. Then I cleaned it up—dropped any rows missing asset types and standardized the text to uppercase. After that, I wrote a function to group similar barrier types together, like lumping anything with "VERTICAL" into one category and "CRACK" into another. Once that was done, I mapped each grouped type to an impact level—HIGH, MEDIUM, or LOW—based on how serious the barrier is for accessibility. I also added a numeric score to make weighting easier later. Finally, I printed out the breakdown to double-check my categories looked right.

In [ ]:
df_barriers = pd.read_csv('ADA_Barriers_in_the_Public_Right_of_Way.csv')

df_barriers = df_barriers.dropna(subset=['ASSET_TYPE'])
df_barriers['ASSET_TYPE'] = df_barriers['ASSET_TYPE'].str.strip().str.upper()

def group_asset(x):
    if 'VERTICAL' in x:
        return 'VERTICAL DISPLACEMENT'
    elif 'HORIZONTAL' in x:
        return 'HORIZONTAL DISPLACEMENT'
    elif 'CRACK' in x or 'CRACKS' in x or 'CRACKED' in x:
        return 'CRACKED SIDEWALK'
    elif 'MISSING' in x or 'END' in x or 'NO SIDEWALK' in x:
        return 'NO SIDEWALK'
    elif 'PAR' in x:
        return 'POOR PAR'
    elif 'POLE' in x:
        return 'POLE'
    elif 'TRASH' in x or 'GARBAGE' in x:
        return 'TRASH'
    elif 'NO WIDTH' in x or 'NO SIDE WITH' in x or 'END' in x:
        return 'POOR WIDTH'
    elif 'SIGN' in x:
        return 'SIGN'
    elif 'TREE' in x or 'GRASS' in x or 'PLANTS' in x or 'VEGETATION' in x or 'DIRT' in x:
        return 'VEGETATION OBSTRUCTION'
    else:
        return 'MISCELLANEOUS'

df_barriers['ASSET_TYPE_GROUPED'] = df_barriers['ASSET_TYPE'].apply(group_asset)

impact_map = {
    'NO SIDEWALK': 'HIGH',
    'POOR WIDTH': 'MEDIUM',
    'VERTICAL DISPLACEMENT': 'HIGH',
    'HORIZONTAL DISPLACEMENT': 'MEDIUM',
    'CRACKED SIDEWALK': 'MEDIUM',
    'POOR PAR': 'MEDIUM',
    'POLE': 'HIGH',
    'SIGN': 'LOW',
    'TRASH': 'LOW',
    'VEGETATION OBSTRUCTION': 'MEDIUM',
    'MISCELLANEOUS': 'MISCELLANEOUS'
}

df_barriers['IMPACT'] = df_barriers['ASSET_TYPE_GROUPED'].map(impact_map)

def assign_impact_score(impact_level):
    scores = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1, 'MISCELLANEOUS': 0}
    return scores.get(impact_level, 0)

df_barriers['IMPACT_SCORE'] = df_barriers['IMPACT'].apply(assign_impact_score)

print("Impact distribution:")
print(df_barriers['IMPACT'].value_counts())
print("\nSample of classified barriers:")
print(df_barriers[['ASSET_TYPE', 'ASSET_TYPE_GROUPED', 'IMPACT']].head(10))

Impact distribution:
IMPACT
HIGH             6715
MEDIUM           4587
MISCELLANEOUS    1088
LOW               414
Name: count, dtype: int64

Sample of classified barriers:
                ASSET_TYPE       ASSET_TYPE_GROUPED  IMPACT
0            SIDEWALK ENDS              NO SIDEWALK    HIGH
1              NO SIDEWALK              NO SIDEWALK    HIGH
2            SIDEWALK ENDS              NO SIDEWALK    HIGH
3            SIDEWALK ENDS              NO SIDEWALK    HIGH
4                     POLE                     POLE    HIGH
5               HORIZONTAL  HORIZONTAL DISPLACEMENT  MEDIUM
6         PAR LESS THAN 4'                 POOR PAR  MEDIUM
7                    GRASS   VEGETATION OBSTRUCTION  MEDIUM
8  HORIZONTAL DISPLACEMENT  HORIZONTAL DISPLACEMENT  MEDIUM
9                     SIGN                     SIGN     LOW


I loaded the ACS GeoJSON file with the disability data. Then I checked to make sure I had the right column—looked for anything with "DP02" in it. Finally, I summed up the disabled population column to get a total and printed it out so I could see the number.

In [ ]:
acs_gdf = gpd.read_file('ACS_5-Year_Social_Characteristics_DC_Ward.geojson')

print(f"Columns containing disability data: {[col for col in acs_gdf.columns if 'DP02' in col]}")
print(f"Total disabled population sum: {acs_gdf['DP02_0072E'].sum():,.0f}")

Columns containing disability data: ['DP02_0001E', 'DP02_0002E', 'DP02_0003E', 'DP02_0004E', 'DP02_0005E', 'DP02_0006E', 'DP02_0007E', 'DP02_0008E', 'DP02_0009E', 'DP02_0010E', 'DP02_0011E', 'DP02_0012E', 'DP02_0013E', 'DP02_0014E', 'DP02_0015E', 'DP02_0016E', 'DP02_0017E', 'DP02_0018E', 'DP02_0019E', 'DP02_0020E', 'DP02_0021E', 'DP02_0022E', 'DP02_0023E', 'DP02_0024E', 'DP02_0025E', 'DP02_0026E', 'DP02_0027E', 'DP02_0028E', 'DP02_0029E', 'DP02_0030E', 'DP02_0031E', 'DP02_0032E', 'DP02_0033E', 'DP02_0034E', 'DP02_0035E', 'DP02_0036E', 'DP02_0037E', 'DP02_0038E', 'DP02_0039E', 'DP02_0040E', 'DP02_0041E', 'DP02_0042E', 'DP02_0043E', 'DP02_0044E', 'DP02_0045E', 'DP02_0046E', 'DP02_0047E', 'DP02_0048E', 'DP02_0049E', 'DP02_0050E', 'DP02_0051E', 'DP02_0052E', 'DP02_0053E', 'DP02_0054E', 'DP02_0055E', 'DP02_0056E', 'DP02_0057E', 'DP02_0058E', 'DP02_0059E', 'DP02_0060E', 'DP02_0061E', 'DP02_0062E', 'DP02_0063E', 'DP02_0064E', 'DP02_0065E', 'DP02_0066E', 'DP02_0067E', 'DP02_0068E', 'DP02_0069E

 Had X and Y coordinate columns in my barriers data, so I used those to create Point geometries—just looped through each row and made a point. Then I turned the whole thing into a GeoDataFrame. The tricky part was figuring out the CRS—I went with EPSG:3857 (Web Mercator) since that's pretty standard. After that, I did a quick sanity check: printed the impact distribution again and looked at a few sample rows to make sure the coordinates and geometry came through right.


In [ ]:
geometry = [Point(x, y) for x, y in zip(df_barriers['X'], df_barriers['Y'])]

barriers_gdf = gpd.GeoDataFrame(df_barriers, geometry=geometry, crs='EPSG:3857')

print("\nImpact level distribution:")
print(barriers_gdf['IMPACT'].value_counts())

print("\nSample of barriers with coordinates:")
print(barriers_gdf[['ASSET_TYPE', 'IMPACT', 'X', 'Y', 'geometry']].head())


Impact level distribution:
IMPACT
HIGH             6715
MEDIUM           4587
MISCELLANEOUS    1088
LOW               414
Name: count, dtype: int64

Sample of barriers with coordinates:
      ASSET_TYPE IMPACT             X             Y  \
0  SIDEWALK ENDS   HIGH -8.575459e+06  4.714795e+06   
1    NO SIDEWALK   HIGH -8.571215e+06  4.698716e+06   
2  SIDEWALK ENDS   HIGH -8.582731e+06  4.712830e+06   
3  SIDEWALK ENDS   HIGH -8.565030e+06  4.707789e+06   
4           POLE   HIGH -8.574380e+06  4.709489e+06   

                           geometry  
0  POINT (-8575459.321 4714794.579)  
1  POINT (-8571215.441 4698716.003)  
2   POINT (-8582731.418 4712829.54)  
3  POINT (-8565030.407 4707788.938)  
4  POINT (-8574380.296 4709488.714)  


 I initially checked the CRS for both datasets—barriers and ACS—to see what I was working with. Turned out they didn't match, so I converted the ACS data to match the barriers' coordinate system. Just a quick reprojection to make sure everything lines up when I do spatial stuff later.


In [ ]:

print(f"Barriers CRS: {barriers_gdf.crs}")
print(f"ACS CRS before: {acs_gdf.crs}")

if acs_gdf.crs != barriers_gdf.crs:
    acs_gdf = acs_gdf.to_crs(barriers_gdf.crs)
    print(f"ACS CRS after conversion: {acs_gdf.crs}")
else:
    print("CRS already matches!")

Barriers CRS: EPSG:3857
ACS CRS before: EPSG:4326
ACS CRS after conversion: EPSG:3857


I took the center point of each census tract—way faster than using the whole polygon boundaries. Then I combined all the barrier points into one object. After that, I calculated the distance from each tract center to the nearest barrier. Ran it, got a range from something like zero up to a few thousand meters.

In [ ]:
acs_gdf['tract_center'] = acs_gdf.geometry.centroid

all_barriers = barriers_gdf.geometry.unary_union

acs_gdf['distance_to_nearest_barrier'] = acs_gdf['tract_center'].apply(lambda x: x.distance(all_barriers))

print("Distance calculation complete!")
print(f"Distance range: {acs_gdf['distance_to_nearest_barrier'].min():.0f} to {acs_gdf['distance_to_nearest_barrier'].max():.0f} meters")

Distance calculation complete!
Distance range: 54 to 381 meters


/tmp/ipykernel_48919/2645563115.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_barriers = barriers_gdf.geometry.unary_union


I calculated two different medians. First, a basic one—each tract counted equally, came out to a certain distance. Then I did a weighted median where tracts with more disabled residents got more say. That one gave me a different number, which makes sense because it's putting more weight on where people actually live.

In [ ]:
median_distance_all = acs_gdf['distance_to_nearest_barrier'].median()
print(f"Median distance to nearest barrier: {median_distance_all:.0f} meters")

def weighted_median(df, value_col, weight_col):
    sorted_df = df.sort_values(value_col)
    cumsum = sorted_df[weight_col].cumsum()
    total = cumsum.iloc[-1]
    median_idx = cumsum.searchsorted(total / 2)
    return sorted_df[value_col].iloc[median_idx]

weighted_median_all = weighted_median(acs_gdf, 'distance_to_nearest_barrier', 'DP02_0072E')
print(f"Population-weighted median distance: {weighted_median_all:.0f} meters")

Median distance to nearest barrier: 160 meters
Population-weighted median distance: 153 meters


Following that, I broke it down by impact level—HIGH, MEDIUM, LOW, and MISCELLANEOUS. For each one, I filtered just those barriers, combined them, then calculated the distance from every tract center to the nearest barrier of that impact type. So now I've got separate distance columns for high-impact barriers, medium, low, and miscellaneous.

In [ ]:
impact_levels = ['HIGH', 'MEDIUM', 'LOW', 'MISCELLANEOUS']

for impact in impact_levels:

    impact_barriers = barriers_gdf[barriers_gdf['IMPACT'] == impact]
    impact_geom = impact_barriers.geometry.unary_union

    col_name = f'distance_to_{impact}'
    acs_gdf[col_name] = acs_gdf['tract_center'].apply(lambda x: x.distance(impact_geom))
    print(f"Calculated distances for {impact} impact barriers")

print("Done!")

Calculated distances for HIGH impact barriers
Calculated distances for MEDIUM impact barriers
Calculated distances for LOW impact barriers
Calculated distances for MISCELLANEOUS impact barriers
Done!


/tmp/ipykernel_48919/1091726834.py:9: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  impact_geom = impact_barriers.geometry.unary_union
/tmp/ipykernel_48919/1091726834.py:9: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  impact_geom = impact_barriers.geometry.unary_union
/tmp/ipykernel_48919/1091726834.py:9: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  impact_geom = impact_barriers.geometry.unary_union
/tmp/ipykernel_48919/1091726834.py:9: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  impact_geom = impact_barriers.geometry.unary_union


I printed out two sets of results. One is the unweighted medians—just straight distances for each impact level. The other is population-weighted, where tracts with more disabled residents carried more weight. That way I could compare the regular average versus what the numbers look like when I factor in where people actually live.

In [ ]:

print("\n=== UNWEIGHTED MEDIANS ===")
for impact in impact_levels:
    col_name = f'distance_to_{impact}'
    median_val = acs_gdf[col_name].median()
    print(f"Median distance to {impact} impact barriers: {median_val:.0f} meters")

print("\n=== POPULATION-WEIGHTED MEDIANS ===")
for impact in impact_levels:
    col_name = f'distance_to_{impact}'
    weighted_median_val = weighted_median(acs_gdf, col_name, 'DP02_0072E')
    print(f"Weighted median distance to {impact} impact barriers: {weighted_median_val:.0f} meters")


=== UNWEIGHTED MEDIANS ===
Median distance to HIGH impact barriers: 186 meters
Median distance to MEDIUM impact barriers: 312 meters
Median distance to LOW impact barriers: 411 meters
Median distance to MISCELLANEOUS impact barriers: 282 meters

=== POPULATION-WEIGHTED MEDIANS ===
Weighted median distance to HIGH impact barriers: 199 meters
Weighted median distance to MEDIUM impact barriers: 235 meters
Weighted median distance to LOW impact barriers: 441 meters
Weighted median distance to MISCELLANEOUS impact barriers: 183 meters


Here I formulated a clean summary table. For each impact level, I pulled together three things: the unweighted median distance, the weighted median distance, and just a count of how many barriers fell into that category. Then I printed it all out nicely so I could see the final breakdown at a glance.

In [ ]:
summary_data = []
for impact in impact_levels:
    col_name = f'distance_to_{impact}'
    summary_data.append({
        'Impact Level': impact,
        'Unweighted Median (m)': round(acs_gdf[col_name].median(), 0),
        'Weighted Median (m)': round(weighted_median(acs_gdf, col_name, 'DP02_0072E'), 0),
        'Number of Barriers': len(barriers_gdf[barriers_gdf['IMPACT'] == impact])
    })

summary_df = pd.DataFrame(summary_data)
print("\n=== FINAL SUMMARY ===")
print(summary_df.to_string(index=False))


=== FINAL SUMMARY ===
 Impact Level  Unweighted Median (m)  Weighted Median (m)  Number of Barriers
         HIGH                  186.0                199.0                6715
       MEDIUM                  312.0                235.0                4587
          LOW                  411.0                441.0                 414
MISCELLANEOUS                  282.0                183.0                1088


I wanted to know, for each tract, what's the impact level of the single closest barrier—not just how far away any barrier is. So I wrote a little function that takes a tract center, measures distances to every single barrier, finds the closest one, and grabs its impact level. Ran that across all tracts. Then I printed out the breakdown to see, for most tracts, whether it's a high, medium, or low impact barrier that's right in their backyard.


In [ ]:
from shapely.geometry import Point

def get_nearest_barrier_impact(row, barriers_gdf):
    tract_center = row['tract_center']

    distances = barriers_gdf.geometry.distance(tract_center)
    min_idx = distances.idxmin()

    return barriers_gdf.loc[min_idx, 'IMPACT']

acs_gdf['nearest_barrier_impact'] = acs_gdf.apply(
    lambda row: get_nearest_barrier_impact(row, barriers_gdf),
    axis=1
)

print("\nImpact level of the CLOSEST barrier to each tract:")
print(acs_gdf['nearest_barrier_impact'].value_counts())


Impact level of the CLOSEST barrier to each tract:
nearest_barrier_impact
HIGH             3
LOW              2
MEDIUM           2
MISCELLANEOUS    1
Name: count, dtype: int64
